# pyads — Pipeline Demo

This notebook walks through the full `pyads` extraction pipeline **offline** — no Mistral API key is needed. A mock LLM client returns a pre-written response so you can explore the normalization, confidence scoring, and benchmark evaluation without making any API calls.

---

### What pyads does

Given a scientific PDF about porous materials (MOFs, COFs, zeolites):

1. **OCR** — uploads the PDF to Mistral's OCR API and saves the extracted text.
2. **Extraction** — sends the OCR text to a Mistral LLM and parses structured adsorption data.
3. **Validation pass** — a stricter second LLM call corrects impossible units and conflated measurements.
4. **Confidence scoring** — compares the two passes field-by-field to flag uncertain values.
5. **CIF download** — searches the Crystallography Open Database (COD) for each material.
6. **CIF analysis** — parses CIF files with gemmi + pymatgen; simulates XRD patterns.

This notebook demonstrates steps 2–4 and the benchmark evaluation.

In [ ]:
# ---------------------------------------------------------------------------
# Setup — add repo root to sys.path so pyads is importable without installing
# ---------------------------------------------------------------------------
import sys
from pathlib import Path

REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repo root: {REPO_ROOT}")

## 1  Load sample OCR text

`data/samples/sample_ocr.txt` contains the OCR output for a published ZIF-8 CO₂/N₂ adsorption paper.  In a real run this text comes from the Mistral OCR API.

In [ ]:
sample_ocr_path = REPO_ROOT / "data" / "samples" / "sample_ocr.txt"
ocr_text = sample_ocr_path.read_text(encoding="utf-8")

print(f"OCR text ({len(ocr_text)} chars):")
print("-" * 60)
print(ocr_text)

## 2  Run first-pass extraction (mocked)

In production `extract_data_from_text` calls the Mistral LLM.  Here we
supply a pre-written JSON response through a mock client so the notebook
runs without an API key.

In [ ]:
import json
from unittest.mock import MagicMock

from pyads.extractor import extract_data_from_text

# --- Mock LLM response (schema v2) ---
MOCK_RESPONSE = json.dumps({
    "doi": "10.1039/c3ce40583f",
    "title": "Selective CO2 capture by ZIF-8 at low partial pressures",
    "year": 2014,
    "materials": [
        {
            "material": "ZIF-8",
            "surface_area": {"value": 1621.0, "unit": "m2/g"},
            "pore_volume":  {"value": 0.636,  "unit": "cm3/g"},
            "pore_size":    {"value": 11.4,   "unit": "A"},
            "gases": ["CO2", "N2"],
            "isotherm_temperatures": [
                {"value": 273, "unit": "K"},
                {"value": 298, "unit": "K"},
                {"value": 77,  "unit": "K"},
            ],
        }
    ],
})

def make_mock_client(response_json):
    msg = MagicMock(); msg.content = response_json
    choice = MagicMock(); choice.message = msg
    usage = MagicMock()
    usage.model_dump.return_value = {"prompt_tokens": 312, "completion_tokens": 87, "total_tokens": 399}
    resp = MagicMock(); resp.choices = [choice]; resp.usage = usage
    client = MagicMock(); client.chat.complete.return_value = resp
    return client

client = make_mock_client(MOCK_RESPONSE)

first_paper, usage1 = extract_data_from_text(
    text=ocr_text,
    source_file="sample_ocr.txt",
    client=client,
    model="mistral-small-latest",
)

print(f"Schema version: {first_paper['schema_version']}")
print(f"Materials found: {len(first_paper['materials'])}")
print(f"Tokens used (mocked): {usage1}")

## 3  Run strict validation pass (mocked)

The validation pass re-reads the evidence with explicit unit rules.  A second mock client simulates the corrected LLM response.

In [ ]:
from pyads.extractor import validate_record_from_text, _attach_material_confidence

# Same values — both passes agree, so confidence will be "high" on all fields
client2 = make_mock_client(MOCK_RESPONSE)

second_paper, usage2 = validate_record_from_text(
    record=first_paper,
    text=ocr_text,
    client=client2,
    model="mistral-small-latest",
)

# Attach per-material confidence (compares first_paper vs second_paper field-by-field)
paper = _attach_material_confidence(first_paper, second_paper)

print("Validation pass complete.")
print(f"Total tokens used (mocked): {usage1.get('total_tokens', 0) + usage2.get('total_tokens', 0)}")

## 4  Inspect the extracted paper (schema v2)

In [ ]:
print(json.dumps(paper, indent=2, ensure_ascii=False))

## 5  Confidence scores

Confidence is computed per material by comparing the two extraction passes:

| Level | Meaning |
|---|---|
| `high` | Both passes agreed on the same non-null value |
| `medium` | Second pass *added* a value the first missed |
| `low` | Passes disagreed — check this field manually |
| `absent` | Field not found in the paper |

In [ ]:
import pandas as pd

for mat in paper["materials"]:
    conf = mat.get("confidence", {})
    print(f"Material: {mat['material']}")
    print(f"  Overall: {conf.get('overall')}")
    for field, score in (conf.get("fields") or {}).items():
        print(f"  {field:<28} {score}")
    print()

## 6  Flatten to a DataFrame (Excel view)

`flatten_record(paper)` returns one row per material — paper-level fields (doi, title, year) are repeated on each row.

In [ ]:
from pyads.extractor import flatten_record

rows = flatten_record(paper)
df = pd.DataFrame(rows)
df

## 7  Benchmark evaluation

`pyads.benchmark` compares extracted results against ground truth and reports per-field precision, recall, and F1.  Here we use the shipped `data/benchmark/ground_truth.json` (5 papers, 6 material records) and the pre-computed `data/samples/sample_adsorption_data.json`.

In [ ]:
from pyads.benchmark import run_evaluation, print_report

extracted_path  = REPO_ROOT / "data" / "samples" / "sample_adsorption_data.json"
ground_truth_path = REPO_ROOT / "data" / "benchmark" / "ground_truth.json"

report = run_evaluation(extracted_path, ground_truth_path)
print_report(report)

## 8  Inspect field metrics as a DataFrame

In [ ]:
metrics_df = pd.DataFrame(report["field_metrics"]).T
metrics_df[["precision", "recall", "f1"]]

## 9  Running the web UI

pyads ships a Streamlit app that provides a point-and-click interface for extraction — no CLI knowledge required.

```bash
pip install streamlit
streamlit run app.py
```

The app opens in your browser at `http://localhost:8501`.  The **Offline demo** tab works without a Mistral API key.

---

## 10  Full pipeline (CLI)

To process real PDFs:

```powershell
# Copy PDFs to data/pdfs/
Copy-Item my_paper.pdf data/pdfs/

# Run OCR only (saves API cost when iterating on extraction)
python runner.py --skip-extraction --skip-cif-download --skip-cif-analysis

# Run extraction with agentic mode (observe–reason–act loop)
python runner.py --skip-ocr --skip-cif-download --skip-cif-analysis --agentic

# Evaluate extraction accuracy against ground truth
python -m pyads.benchmark data/extracted/adsorption_data.json
```